# Conveyor perception

**An end-to-end industrial CV pipeline on a free T4:** real recycling data → trained model → live detection → drift monitoring → triage decisions.

- **Runtime:** Google Colab T4 (free tier, ~12h cap).  
- **Data:** bundled 4-class recycling set (CC BY 4.0).  
- **Model:** YOLO26s, trained in-kernel, cached on re-run.  
- **Goal:** show the loop — train → infer → drift → triage → maintain — on real data, in <5 minutes.


In [ ]:
# --- Cell 1: Runtime + env check ---
import os, sys, json, platform
from pathlib import Path

# --- 1. Colab or local? ---
IN_COLAB = 'google.colab' in sys.modules
print(f'  Runtime: {"Google Colab" if IN_COLAB else "Local (" + platform.node() + ")"}')

# --- 2. Python + key libs (skip import if missing) ---
print(f'  Python: {sys.version.split()[0]}  ({sys.executable.split("/")[-1]})')
for mod in ['numpy', 'torch', 'ultralytics', 'supervision', 'roboflow']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'  {mod:14s} {v}')
    except ImportError:
        print(f'  {mod:14s} — not installed yet (cell 2 will install)')

# --- 3. GPU (or warn if CPU-only) ---
_gpu = 'unknown'
try:
    import torch
    _gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
except Exception:
    pass
print(f'  GPU:    {_gpu}')

# --- 4. Disk + RAM ---
_disk_free = '?'
try:
    import shutil
    _u = shutil.disk_usage('/')
    _disk_free = f'{_u.free / 1e9:.1f} GB free of {_u.total / 1e9:.1f} GB'
except Exception:
    pass
print(f'  Disk:   {_disk_free}')

# --- 5. Locate the repo (for local runs) — Colab gets cloned by cell 2 ---
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
if not IN_COLAB:
    # Walk up until we find the repo root (contains pyproject.toml)
    while not (REPO / 'pyproject.toml').exists() and REPO != REPO.parent:
        REPO = REPO.parent
print(f'  Repo:   {REPO}{" (will be cloned here by cell 2)" if IN_COLAB else ""}')

# NOTE: colab_session + the state singleton are intentionally NOT used here.
# Cell 1 runs BEFORE the clone (cell 2), so the repo isn't on disk yet — any
# `import colab_session` would crash with ModuleNotFoundError. State init is
# deferred to cell 3, which runs after the clone + install are done. (Aug 22 2026)
print()
print('  ✓ env check done.  Next: cell 2 (clone + install).')


In [ ]:
# --- Cell 2: Install + clone ---
import os, sys, subprocess, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Clone repo (idempotent — skip if pyproject.toml already there) ---
if IN_COLAB:
    if (REPO / 'pyproject.toml').exists():
        print(f'  Repo already at {REPO} (skipping clone)')
    else:
        print(f'  Cloning conveyor-perception -> {REPO}...')
        REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/roniejosephv-star/conveyor-perception.git',
             str(REPO)],
            check=True,
        )
        print('  ✓ cloned.')
else:
    print(f'  Local repo at {REPO} (skipping clone)')

# --- 2. Add REPO + REPO/notebooks to sys.path (colab_session.py lives in notebooks/) ---
for _p in (REPO, REPO / 'notebooks'):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)
print(f'  Path:  {REPO}  +  {REPO}/notebooks')

# --- 3. Install minimal open-source deps (numpy/torch already on Colab) ---
# 'trackers>=2.6.0' is for the ByteTrack tracker in TrackingPipeline. Without
# it, the pipeline falls back to a simple IoU tracker (less robust to occlusion).
INSTALL = ['ultralytics', 'supervision', 'opencv-python-headless', 'roboflow', 'trackers>=2.6.0']
print(f'  Installing: {", ".join(INSTALL)}')
subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', *INSTALL]
)
print('  ✓ installed.')

# --- 4. Verify the key imports work (smoke test) ---
print('  Verifying:')
for _mod in ['ultralytics', 'supervision', 'roboflow']:
    try:
        _m = importlib.import_module(_mod)
        _v = getattr(_m, '__version__', '?')
        print(f'    {_mod:14s} {_v}  ok')
    except ImportError as _e:
        print(f'    {_mod:14s} FAIL  {_e}')

# --- 5. Verify colab_session is now importable (the import cell 3 needs) ---
try:
    from colab_session import get_state
    print('    colab_session  ok  (get_state() ready for cell 3)')
except ImportError as _e:
    print(f'    colab_session  FAIL  {_e}')

# --- 6. Quick sanity check: list the repo top-level ---
print(f'  Repo top-level:')
for _entry in sorted(REPO.iterdir()):
    if not _entry.name.startswith('.'):
        print(f'    {_entry.name}')

print()
print('  ✓ install + clone done.  Next: cell 3 (state + toggles).')


In [ ]:
# --- Cell 3: State + toggles ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Defensive install: ipywidgets is needed for the toggle UI ---
# (Colab has it pre-installed, but cell 3 should not assume cell 2 installed it.)
try:
    import ipywidgets  # noqa: F401
    print('  ipywidgets: ok')
except ImportError:
    print('  Installing ipywidgets (toggle UI dep)...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', 'ipywidgets']
    )
    print('  ✓ installed.')

# --- 2. Idempotent sys.path setup (cell 2 already did this; redo is harmless) ---
for _p in (REPO, REPO / 'notebooks'):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 3. Create the SessionState singleton (lives in builtins so every cell sees it) ---
from colab_session import get_state, toggle_ui

state = get_state()
state.log('cell-3', action='init', note='state singleton + toggle UI ready')

# --- 4. Show the toggle UI (4 framework abstractions + 8 JD modules) ---
# Untick anything you want the pipeline to skip. Defaults: all enabled.
print()
print('  Toggle the components below — defaults are all enabled.')
print('  Untick anything you want to skip in the pipeline.')
print()
ui = toggle_ui()
display(ui)

# --- 5. Print a summary of what's currently enabled (grouped) ---
print()
print('  Current toggles:')
abstr_keys = [k for k in state.toggles if k.startswith('abstraction:')]
mod_keys = [k for k in state.toggles if k.startswith('module:')]
print('    4 framework abstractions:')
for _k in abstr_keys:
    _v = state.toggles[_k]
    _icon = '✓' if _v else '○'
    print(f'      {_icon} {_k}')
print('    8 JD modules:')
for _k in mod_keys:
    _v = state.toggles[_k]
    _icon = '✓' if _v else '○'
    print(f'      {_icon} {_k}')

_n_on = sum(state.toggles.values())
_n_total = len(state.toggles)
print()
print(f'  ✓ state + toggles ready.  {_n_on}/{_n_total} components enabled.')
print('  Next: cell 4 (load abstractions).')


In [ ]:
# --- Cell 4: Load abstractions ---
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# --- 1. Path setup: src/ (for `import conveyor_perception`) + REPO/notebooks ---
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 2. Import the 4 abstraction classes (unconditional — fail fast on missing module) ---
from colab_session import get_state
from conveyor_perception.core.detection_pipeline import DetectionPipeline as Detector
from conveyor_perception.core.tracking_pipeline import TrackingPipeline
from conveyor_perception.core.drift_monitor import DriftMonitor
from conveyor_perception.core.triage_surface import MCPTriageSurface, InMemoryAlertQueue

state = get_state()
loaded: dict = {}
skipped: list = []

# --- 3. Toggle-gated load (the toggle UI from cell 3 controls this) ---
if state.toggles.get('abstraction:detector', True):
    # Detector is just a class ref for now — model is wired in cell 8 after training.
    loaded['detector'] = Detector
    print('  ✓ Detector class loaded (YOLO26 + OpenCV DNN)')
else:
    skipped.append('abstraction:detector')
    print('  ○ Detector skipped (toggle off)')

if state.toggles.get('abstraction:tracker', True):
    loaded['tracker'] = TrackingPipeline()
    print('  ✓ TrackingPipeline instantiated (ByteTrack)')
else:
    skipped.append('abstraction:tracker')
    print('  ○ TrackingPipeline skipped (toggle off)')

if state.toggles.get('abstraction:drift_monitor', True):
    loaded['drift_monitor'] = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
    print('  ✓ DriftMonitor instantiated (KS test + z-score + MAD)')
else:
    skipped.append('abstraction:drift_monitor')
    print('  ○ DriftMonitor skipped (toggle off)')

if state.toggles.get('abstraction:triage', True):
    loaded['triage_surface'] = MCPTriageSurface('l1-triage', InMemoryAlertQueue())
    print('  ✓ MCPTriageSurface instantiated (5 MCP tools)')
else:
    skipped.append('abstraction:triage')
    print('  ○ MCPTriageSurface skipped (toggle off)')

state.log('cell-4', action='load-abstractions', loaded=list(loaded.keys()), skipped=skipped)
print()
print(f'  ✓ loaded {len(loaded)}/4 abstractions, skipped {len(skipped)}.')
print('  Next: cell 5 (load modules).')


In [ ]:
# --- Cell 5: Load modules ---
import os, sys, importlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# --- 1. Path setup (idempotent — cell 4 already added REPO/src) ---
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 2. Module metadata: toggle key + import path + 1-line description ---
from colab_session import get_state
state = get_state()

MODULES_META = [
    ('module:perception',             'conveyor_perception.perception',             'UltralyticsDetector + RecyclingInferenceService'),
    ('module:triage',                 'conveyor_perception.triage',                 'L1TriageAgent + 7 severity rules'),
    ('module:predictive_maintenance', 'conveyor_perception.predictive_maintenance', 'MaintenanceAdvisor + 3 signal types'),
    ('module:multitask',              'conveyor_perception.multitask',              'MultitaskPipeline (Detector->Tracker->Drift->Triage)'),
    ('module:integration',            'conveyor_perception.integration',            'ConveyorNode (real ROS 2) + MockROS2Node (CI)'),
    ('module:robustness',             'conveyor_perception.robustness',             'RobustnessTestSuite + 13 augmentations'),
    ('module:monitoring',             'conveyor_perception.monitoring',             'MonitoringDashboard + ShiftReport'),
    ('module:optimization',           'conveyor_perception.optimization',           'benchmark_pytorch/onnx + export_onnx'),
]

loaded: list = []
skipped: list = []
failed: list = []

# --- 3. Toggle-gated dynamic load (one bad module doesn't kill the cell) ---
print('  Loading 8 JD modules:')
for _toggle_key, _module_path, _desc in MODULES_META:
    if not state.toggles.get(_toggle_key, True):
        skipped.append(_toggle_key)
        print(f'    ○ {_module_path}  (disabled by toggle)')
        continue
    try:
        importlib.import_module(_module_path)
        loaded.append(_module_path)
        print(f'    ✓ {_module_path}')
        print(f'        {_desc}')
    except Exception as _e:
        failed.append((_module_path, str(_e)))
        print(f'    ✗ {_module_path}  failed: {_e}')

state.log(
    'cell-5',
    action='load-modules',
    loaded=loaded,
    skipped=skipped,
    failed=[m for m, _ in failed],
)
print()
print(f'  ✓ loaded {len(loaded)}/{len(MODULES_META)} modules, skipped {len(skipped)}, failed {len(failed)}.')
print('  Next: cell 6 (data registry).')


In [ ]:
# --- Cell 6: Data registry ---
import os, sys, subprocess, json
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

# --- 1. Defensive PyYAML install (data.yaml is YAML) ---
try:
    import yaml  # noqa: F401
except ImportError:
    print('  Installing PyYAML (data.yaml parser)...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-input', 'pyyaml']
    )
    print('  ✓ installed.')
    import yaml

# --- 2. Path setup (idempotent) ---
SRC = REPO / 'src'
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

# --- 3. Scan data/ for datasets with data.yaml ---
from colab_session import get_state
state = get_state()

DATA_ROOTS = [REPO / 'data' / 'sample', REPO / 'data' / 'raw']

def _count_imgs(_d: Path, _rel_path: str) -> int:
    if not _rel_path:
        return 0
    _p = _d / _rel_path.lstrip('./').lstrip('/')
    if not _p.exists():
        return 0
    return len(list(_p.glob('*.jpg'))) + len(list(_p.glob('*.png')))

def _dir_size_mb(_d: Path) -> float:
    if not _d.exists():
        return 0.0
    return sum(f.stat().st_size for f in _d.rglob('*') if f.is_file()) / (1024 * 1024)

registry: list = []
for _root in DATA_ROOTS:
    if not _root.exists():
        continue
    for _d in sorted(_root.iterdir()):
        if not _d.is_dir():
            continue
        _yaml_p = _d / 'data.yaml'
        if not _yaml_p.exists():
            continue
        try:
            _cfg = yaml.safe_load(_yaml_p.read_text())
        except Exception as _e:
            print(f'  ✗ could not parse {_yaml_p}: {_e}')
            _cfg = {}
        _nc = _cfg.get('nc', len(_cfg.get('names', [])))
        _names = _cfg.get('names', [])
        _n_train = _count_imgs(_d, _cfg.get('train', 'train/images'))
        _n_val = _count_imgs(_d, _cfg.get('val', 'val/images'))
        _meta_p = _d / 'dataset_meta.json'
        _meta = json.loads(_meta_p.read_text()) if _meta_p.exists() else {}
        _status = 'bundled' if _root == REPO / 'data' / 'sample' else 'downloaded'
        _size_mb = _dir_size_mb(_d)
        registry.append({
            'name': _d.name,
            'path': str(_d),
            'status': _status,
            'n_train': _n_train,
            'n_val': _n_val,
            'nc': _nc,
            'names': _names,
            'source': _meta.get('source', '—'),
            'license': _meta.get('license', '—'),
            'baseline_mAP50': _meta.get('pre_trained_baseline_mAP50'),
            'size_mb': _size_mb,
        })

# --- 4. Cache the registry to state (downstream cells read it) ---
state.dataset_registry = registry
state.metric('datasets_available', len(registry))
state.log(
    'cell-6',
    action='data-registry',
    n_datasets=len(registry),
    names=[_r['name'] for _r in registry],
)

# --- 5. Render as a text table ---
W = 82
print()
print('─' * W)
print(f'  DATA REGISTRY  ({len(registry)} dataset(s) on disk)'.center(W))
print('─' * W)
if not registry:
    print('  (no datasets found — run cell 7 to download the recycling_v3 set)')
else:
    for _r in registry:
        print(f'  • {_r["name"]}  [{_r["status"]}]')
        print(f'      path:    {_r["path"]}')
        print(f'      images:  {_r["n_train"]} train  /  {_r["n_val"]} val  /  {_r["size_mb"]:.1f} MB')
        print(f'      classes ({_r["nc"]}): {
.join(str(_n) for _n in _r["names"])}')
        if _r.get('source') and _r['source'] != '—':
            print(f'      source:  {_r["source"]}')
        if _r.get('license') and _r['license'] != '—':
            print(f'      license: {_r["license"]}')
        if _r.get('baseline_mAP50') is not None:
            print(f'      baseline mAP50: {_r["baseline_mAP50"]}')
        print()
print('─' * W)
if registry:
    print(f'  ✓ {len(registry)} dataset(s) registered.  Next: cell 7 (download if you need a bigger one).')
else:
    print('  ✓ registry ready (empty).  Next: cell 7 (download).')


In [ ]:
# --- Cell 7: Data download (idempotent) ---
import os, sys, shutil
from pathlib import Path
from colab_session import get_state

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
DATA_RAW = REPO / 'data' / 'raw'

# Change TARGET_NAME to download a different dataset (or 'skip' to do nothing).
TARGET_NAME = 'recycling_v3'  # 'recycling_v3' | 'everyday_recycle_waste' | 'recycling_classification' | 'skip'

DATASETS = {
    'recycling_v3':             dict(workspace='zkf624',                       project='-recycling',               version=3),
    'everyday_recycle_waste':   dict(workspace='everyday-recycle-waste-xhayk', project='everyday-recycle-waste',   version=3),
    'recycling_classification': dict(workspace='new-workspace-eosax',          project='recycling-classification', version=1),
}

SAFE_NAME = TARGET_NAME.replace('-', '_')
TARGET = DATA_RAW / SAFE_NAME
DATA_YAML = TARGET / 'data.yaml'

def _count_imgs(_p: Path) -> int:
    if not _p.exists():
        return 0
    return len(list(_p.glob('*.jpg'))) + len(list(_p.glob('*.png')))

def _dir_size_mb(_d: Path) -> float:
    if not _d.exists():
        return 0.0
    return sum(f.stat().st_size for f in _d.rglob('*') if f.is_file()) / (1024 * 1024)

W = 82
print()
print('─' * W)
print(f'  DOWNLOAD CHECK  —  target: {SAFE_NAME}'.center(W))
print('─' * W)

if DATA_YAML.exists():
    # 1. Already on disk — skip the download (idempotent re-run path).
    n_train = _count_imgs(TARGET / 'train' / 'images')
    n_val = _count_imgs(TARGET / 'val' / 'images')
    n_test = _count_imgs(TARGET / 'test' / 'images')
    size_mb = _dir_size_mb(TARGET)
    print(f'  ✓ {SAFE_NAME} already on disk — download skipped')
    print()
    print(f'    train:  {n_train:>6} images')
    print(f'    val:    {n_val:>6} images' + (f'   (incl. test: {n_test})' if n_test else ''))
    print(f'    size:   {size_mb:>7.1f} MB on disk')
    print(f'    path:   {TARGET}')
    print('─' * W)
    state.metric(f'dataset_{SAFE_NAME}_status', 'already_present')
    state.metric(f'dataset_{SAFE_NAME}_n_train', n_train)
    state.metric(f'dataset_{SAFE_NAME}_n_val', n_val)
    state.metric(f'dataset_{SAFE_NAME}_size_mb', round(size_mb, 1))
    state.log(
        'cell-7',
        action='skip',
        reason='already-present',
        dataset=SAFE_NAME,
        n_train=n_train,
        n_val=n_val,
        size_mb=round(size_mb, 1),
    )
elif TARGET_NAME == 'skip':
    print('  TARGET_NAME=skip — no download attempted.')
    print('─' * W)
    state.metric('dataset_download_status', 'skipped')
else:
    # 2. Try Roboflow Universe (public demo key, read-only for CC-BY 4.0 datasets).
    cfg = DATASETS.get(SAFE_NAME)
    if cfg is None:
        print(f'  ✗ Unknown dataset: {TARGET_NAME!r}')
        print(f'  Add an entry to DATASETS in this cell, then re-run.')
        print('─' * W)
        state.metric(f'dataset_{SAFE_NAME}_status', 'unknown')
    else:
        print(f'  Downloading {SAFE_NAME} from Roboflow Universe...')
        print(f'    workspace: {cfg["workspace"]}')
        print(f'    project:   {cfg["project"]}')
        print(f'    version:   {cfg["version"]}')
        print('─' * W)
        _ok = False
        _err = None
        try:
            from roboflow import Roboflow
            _api_key = 'qogO5hAuLgUUYMbNT6W3'  # public demo key, read-only
            _env_p = REPO / '.env'
            if _env_p.exists():
                for _line in _env_p.read_text().splitlines():
                    if _line.startswith('ROBOFLOW_API_KEY='):
                        _api_key = _line.split('=', 1)[1].strip() or _api_key
            _rf = Roboflow(api_key=_api_key)
            _project = _rf.workspace(cfg['workspace']).project(cfg['project'])
            _version = _project.version(cfg['version'])
            # 'yolov11' is the Roboflow LABEL format (YOLO .txt annotations
            # + folder layout), NOT the model architecture. The YOLO .txt schema
            # is identical across YOLOv5/8/11/26 — the actual model choice
            # (yolo26s) happens in cell 8 via YOLO('yolo26s.pt').
            _dataset = _version.download('yolov11')
            _downloaded = Path(_dataset.location)
            if _downloaded.exists():
                DATA_RAW.mkdir(parents=True, exist_ok=True)
                if TARGET.exists():
                    shutil.rmtree(TARGET)
                _downloaded.rename(TARGET)
                _ok = True
        except ImportError as _e:
            _err = f'roboflow not installed ({_e})'
        except Exception as _e:
            _err = f'{type(_e).__name__}: {_e}'

        if _ok:
            n_train = _count_imgs(TARGET / 'train' / 'images')
            n_val = _count_imgs(TARGET / 'val' / 'images')
            size_mb = _dir_size_mb(TARGET)
            print(f'  ✓ Downloaded {SAFE_NAME}')
            print()
            print(f'    train:  {n_train:>6} images')
            print(f'    val:    {n_val:>6} images')
            print(f'    size:   {size_mb:>7.1f} MB on disk')
            print(f'    path:   {TARGET}')
            print('─' * W)
            state.metric(f'dataset_{SAFE_NAME}_status', 'downloaded')
            state.metric(f'dataset_{SAFE_NAME}_n_train', n_train)
            state.metric(f'dataset_{SAFE_NAME}_n_val', n_val)
            state.metric(f'dataset_{SAFE_NAME}_size_mb', round(size_mb, 1))
            state.log(
                'cell-7',
                action='downloaded',
                dataset=SAFE_NAME,
                n_train=n_train,
                n_val=n_val,
                size_mb=round(size_mb, 1),
            )
        else:
            print(f'  ✗ Download failed: {_err}')
            print()
            print('  Manual fallback:')
            print(f'    1. Open https://universe.roboflow.com/{cfg["workspace"]}/{cfg["project"]}/dataset/{cfg["version"]}')
            print('    2. Click Download Dataset -> Format: YOLOv11')
            print(f'    3. Extract the zip into {DATA_RAW}/ so the path is:')
            print(f'       {DATA_RAW}/{SAFE_NAME}/data.yaml')
            print('─' * W)
            state.metric(f'dataset_{SAFE_NAME}_status', 'failed')
            state.log('cell-7', action='download-failed', error=str(_err), dataset=SAFE_NAME)

# 3. Refresh the registry on state (so cell 8 sees the new dataset).
import yaml as _yaml
_registry = []
for _root in [REPO / 'data' / 'sample', REPO / 'data' / 'raw']:
    if not _root.exists():
        continue
    for _d in sorted(_root.iterdir()):
        if not _d.is_dir():
            continue
        _yp = _d / 'data.yaml'
        if not _yp.exists():
            continue
        try:
            _cfg = _yaml.safe_load(_yp.read_text())
        except Exception:
            _cfg = {}
        _nc = _cfg.get('nc', len(_cfg.get('names', [])))
        _names = _cfg.get('names', [])
        _n_train = _count_imgs(_d / _cfg.get('train', 'train/images').lstrip('./').lstrip('/'))
        _n_val = _count_imgs(_d / _cfg.get('val', 'val/images').lstrip('./').lstrip('/'))
        _status = 'bundled' if _root == REPO / 'data' / 'sample' else 'downloaded'
        _size_mb = _dir_size_mb(_d)
        _registry.append({
            'name': _d.name,
            'path': str(_d),
            'status': _status,
            'n_train': _n_train,
            'n_val': _n_val,
            'nc': _nc,
            'names': _names,
            'size_mb': _size_mb,
        })
state.dataset_registry = _registry
state.metric('datasets_available', len(_registry))
print()
print(f'  ✓ {len(_registry)} dataset(s) now in registry.  Next: cell 8 (train).')


In [ ]:
# --- Cell 8: Train (cached on re-run) ---
import os, sys, time, csv
from pathlib import Path
from colab_session import get_state, pick_device

IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
state = get_state()

# --- 0. Pick the dataset (change DATASET_NAME to switch) ---
DATASET_NAME = 'recycling_demo'  # 'recycling_demo' (bundled, ~1 min) or 'recycling_v3' (downloaded, ~5-10 min)

# --- 1. Resolve the dataset path from the registry (or fall back to disk scan) ---
_reg = getattr(state, 'dataset_registry', None) or []
_match = next((_r for _r in _reg if _r['name'] == DATASET_NAME), None)
if _match is None:
    for _root in [REPO / 'data' / 'sample', REPO / 'data' / 'raw']:
        _cand = _root / DATASET_NAME
        if (_cand / 'data.yaml').exists():
            _match = {'name': DATASET_NAME, 'path': str(_cand), 'n_train': 0, 'n_val': 0, 'nc': 0, 'names': []}
            break

# --- 2. Compute the cache + toggle gate (skip = don't crash) ---
_skip_reason = None
if not state.toggles.get('module:perception', True):
    _skip_reason = 'toggle-off'
elif _match is None:
    _skip_reason = 'dataset-not-found'

if _skip_reason:
    print('─' * 72)
    if _skip_reason == 'toggle-off':
        print('  module:perception toggle is OFF — skipping training.')
        print('  Re-enable the toggle in cell 3 to train the model.')
    else:
        print(f'  ✗ Dataset {DATASET_NAME!r} not found in registry or on disk.')
        print('  Run cell 6 to scan, or cell 7 to download.')
    print('─' * 72)
    state.log('cell-8', action='skipped', reason=_skip_reason, dataset=DATASET_NAME)
    print('  Next: cell 9 (compare).')
else:
    DATA_DIR = Path(_match['path'])
    YAML_P = DATA_DIR / 'data.yaml'
    MODEL_DIR = REPO / 'models' / DATASET_NAME
    BEST_PT = MODEL_DIR / 'weights' / 'best.pt'
    RESULTS_CSV = MODEL_DIR / 'results.csv'

    print('─' * 72)
    _n_train = _match.get('n_train', 0) or 0
    _n_val = _match.get('n_val', 0) or 0
    _nc = _match.get('nc', 0) or 0
    print(f'  TRAIN  —  dataset: {DATASET_NAME}  (train: {_n_train}, val: {_n_val}, cls: {_nc})'.center(72))
    print('─' * 72)

    if BEST_PT.exists() and RESULTS_CSV.exists():
        # --- Cached path: read metrics from results.csv, no re-training ---
        print(f'  ✓ Cached model found at {BEST_PT}')
        print(f'      size: {BEST_PT.stat().st_size / 1e6:.1f} MB')
        try:
            with open(RESULTS_CSV) as _f:
                _rows = list(csv.DictReader(_f))
                _rows = [_r for _r in _rows if any(v.strip() for v in _r.values())]
            if _rows:
                _last = _rows[-1]
                _strip = lambda _k: _last.get(_k, '').strip()
                print()
                print('  -- Final epoch metrics (from results.csv) --')
                for _k in [
                    'epoch', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss',
                    'metrics/precision(B)', 'metrics/recall(B)',
                    'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'lr/pg0',
                ]:
                    if _k in _last:
                        print(f'      {_k:30}  {_strip(_k)}')
                try:
                    _map50 = float(_strip('metrics/mAP50(B)'))
                    state.metric(f'map50_{DATASET_NAME}', _map50)
                    state.active_model_path = str(BEST_PT)
                    state.active_dataset = DATASET_NAME
                    state.log('cell-8', action='cached', dataset=DATASET_NAME, mAP50=_map50)
                except Exception:
                    pass
        except Exception as _e:
            print(f'  could not read results.csv: {_e}')
        print()
        print(f'  re-run skipped: cached model used. (delete {MODEL_DIR} to retrain.)')
        print('─' * 72)
    else:
        # --- Fresh path: actually train ---
        print(f'  No cached model — training YOLO26s on {DATASET_NAME}...')
        print('  (first run on a dataset: 1-15 min depending on size; later runs are cached)')
        t0 = time.time()
        try:
            from ultralytics import YOLO
            import torch
            device = pick_device()
            gpu_name = torch.cuda.get_device_name(0) if device == '0' else 'cpu'
            print(f'  device: {device} ({gpu_name})')
            print(f'  data.yaml: {YAML_P}')

            _epochs = 8 if _n_train < 200 else 30
            print(f'  epochs: {_epochs}  (auto-sized for {_n_train} train images)')

            model = YOLO('yolo26s.pt')
            # patience=3 (not the Ultralytics default of 15): for short runs
            # like our 8-epoch demo on recycling_demo, patience=15 never fires
            # because the epoch cutoff happens first. patience=3 gives early
            # stopping teeth on small data while still being forgiving on
            # 30-epoch runs on recycling_v3.
            model.train(
                data=str(YAML_P),
                epochs=_epochs,
                imgsz=640,
                batch=16,
                device=device,
                project=str(MODEL_DIR.parent),
                name=DATASET_NAME,
                exist_ok=True,
                patience=3,
                verbose=True,
                plots=False,
            )
            train_time = time.time() - t0
            state.metric(f'train_time_{DATASET_NAME}', round(train_time, 1))
            print(f'\n  Training complete in {train_time/60:.1f} min')
            if BEST_PT.exists():
                print(f'    best.pt: {BEST_PT} ({BEST_PT.stat().st_size / 1e6:.1f} MB)')
                state.active_model_path = str(BEST_PT)
                state.active_dataset = DATASET_NAME
                try:
                    with open(RESULTS_CSV) as _f:
                        _last = list(csv.DictReader(_f))[-1]
                    _map50 = float(_last.get('metrics/mAP50(B)', '0').strip())
                    state.metric(f'map50_{DATASET_NAME}', _map50)
                    print(f'    mAP50: {_map50:.3f}')
                    state.log(
                        'cell-8',
                        action='trained',
                        dataset=DATASET_NAME,
                        mAP50=_map50,
                        train_time=round(train_time, 1),
                    )
                except Exception as _e:
                    print(f'    (could not read final mAP from results.csv: {_e})')
                    state.log(
                        'cell-8',
                        action='trained',
                        dataset=DATASET_NAME,
                        train_time=round(train_time, 1),
                    )
        except Exception as _e:
            print(f'  ✗ Training failed: {type(_e).__name__}: {_e}')
            state.log(
                'cell-8',
                action='failed',
                reason='training-error',
                error=str(_e),
                dataset=DATASET_NAME,
            )
        print()
        print('─' * 72)
        print('  Trained. To re-run for cached output, re-run this cell.')
        print(f'  To force retrain:  !rm -rf {MODEL_DIR}  then re-run this cell.')
        print('─' * 72)
    print('  Next: cell 9 (compare).')


In [ ]:
# --- Cell 9: Compare trained models (side-by-side) ---
import csv
from pathlib import Path
from colab_session import get_state

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
MODELS_ROOT = REPO / 'models'

W = 100
print('─' * W)
print(f'  MODEL COMPARISON  (side-by-side metrics)'.center(W))
print('─' * W)

rows: list = []
if MODELS_ROOT.exists():
    for _md in sorted(MODELS_ROOT.iterdir()):
        if not _md.is_dir():
            continue
        _best_pt = _md / 'weights' / 'best.pt'
        _results_csv = _md / 'results.csv'
        if not (_best_pt.exists() and _results_csv.exists()):
            continue
        try:
            with open(_results_csv) as _f:
                _csv_rows = [r for r in csv.DictReader(_f) if any(v.strip() for v in r.values())]
            if not _csv_rows:
                continue
            _last = _csv_rows[-1]
            _s = lambda _k: _last.get(_k, '').strip()
            rows.append({
                'name': _md.name,
                'path': str(_best_pt),
                'size_mb': _best_pt.stat().st_size / 1e6,
                'epochs': _s('epoch'),
                'precision': _s('metrics/precision(B)'),
                'recall': _s('metrics/recall(B)'),
                'mAP50': _s('metrics/mAP50(B)'),
                'mAP50_95': _s('metrics/mAP50-95(B)'),
            })
        except Exception as _e:
            print(f'  could not read {_md.name}: {_e}')

if not rows:
    print('  No trained models found. Run cell 8 at least once.')
    state.log('cell-9', action='empty', reason='no-models')
else:
    print(f'  {"NAME":18}  {"SIZE":>6}  {"EPOCHS":>6}  {"P":>7}  {"R":>7}  {"mAP50":>7}  {"mAP50-95":>9}')
    print('─' * W)
    for _r in rows:
        print(f'  {_r["name"]:18}  {_r["size_mb"]:>5.1f}M  {_r["epochs"]:>6}  '
              f'{_r["precision"]:>7}  {_r["recall"]:>7}  {_r["mAP50"]:>7}  {_r["mAP50_95"]:>9}')
    print('─' * W)

    if len(rows) >= 2:
        _best = max(rows, key=lambda _r: float(_r['mAP50']) if _r['mAP50'] else 0)
        print(f'  Best mAP50:  {_best["name"]}  ({_best["mAP50"]})')
        # Promote the best model to active_model_path so downstream cells use it.
        state.active_model_path = _best['path']
        state.active_dataset = _best['name']
        state.metric('best_mAP50_model', _best['name'])
        state.metric('best_mAP50_value', float(_best['mAP50']))
    elif len(rows) == 1:
        _only = rows[0]
        print(f'  Only 1 model: {_only["name"]}  (mAP50={_only["mAP50"]})')
        state.active_model_path = _only['path']
        state.active_dataset = _only['name']
        state.metric('best_mAP50_model', _only['name'])
        if _only['mAP50']:
            state.metric('best_mAP50_value', float(_only['mAP50']))
    print('─' * W)

    state.log(
        'cell-9',
        action='compare',
        n_models=len(rows),
        names=[_r['name'] for _r in rows],
    )

    # Optional bar chart (matplotlib is a Colab standard dep; defensive try/except).
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        _names = [_r['name'] for _r in rows]
        _map50 = [float(_r['mAP50']) if _r['mAP50'] else 0 for _r in rows]
        _map50_95 = [float(_r['mAP50_95']) if _r['mAP50_95'] else 0 for _r in rows]
        _x = list(range(len(rows)))
        _w = 0.35
        _fig, _ax = plt.subplots(figsize=(7, 4))
        _b1 = _ax.bar([_i - _w/2 for _i in _x], _map50, _w, label='mAP50', color='#22C55E')
        _b2 = _ax.bar([_i + _w/2 for _i in _x], _map50_95, _w, label='mAP50-95', color='#3B82F6')
        _ax.set_xticks(_x)
        _ax.set_xticklabels(_names, rotation=20, ha='right')
        _ax.set_ylabel('mAP')
        _ax.set_ylim(0, 1.0)
        _ax.set_title('Trained models — mAP comparison')
        _ax.legend()
        _ax.grid(axis='y', alpha=0.3)
        for _bar in list(_b1) + list(_b2):
            _h = _bar.get_height()
            _ax.text(_bar.get_x() + _bar.get_width()/2, _h + 0.01, f'{_h:.2f}', ha='center', fontsize=9)
        plt.tight_layout()
        _chart_path = REPO / 'comparison.png'
        plt.savefig(str(_chart_path), dpi=100)
        print(f'  chart saved to {_chart_path}')
    except ImportError:
        print('  (matplotlib not available — text table is the source of truth)')
    except Exception as _e:
        print(f'  (chart skipped: {type(_e).__name__}: {_e})')

print()
print('  ✓ compare done.  Next: cell 10 (pipeline).')


In [ ]:
# --- Cell 10: Pipeline (Detector→Tracker→Drift→Triage→Maintenance) ---
import os, sys, time
from pathlib import Path
from colab_session import get_state, pick_device

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# Idempotent path setup (REPO + REPO/notebooks + REPO/src)
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

W = 82
print('─' * W)
print('  PIPELINE  (Detector→Tracker→Drift→Triage→Maintenance)'.center(W))
print('─' * W)

# --- 0. Skip if no active model or toggle off ---
_skip_reason = None
if not getattr(state, 'active_model_path', None):
    _skip_reason = 'no-model'
elif not Path(state.active_model_path).exists():
    _skip_reason = 'model-missing'
elif not state.toggles.get('module:multitask', True):
    _skip_reason = 'toggle-off'

if _skip_reason:
    if _skip_reason == 'no-model':
        print('  No active model. Run cell 8 to train (cell 9 picks the best).')
    elif _skip_reason == 'model-missing':
        print(f'  Active model missing: {state.active_model_path}')
        print('  Re-run cell 8 to re-train.')
    else:
        print('  module:multitask toggle is OFF — skipping pipeline.')
        print('  Re-enable the toggle in cell 3 to run the pipeline.')
    print('─' * W)
    state.log('cell-10', action='skipped', reason=_skip_reason)
    print('  Next: cell 11 (visual analytics).')
else:
    print(f'  Active model:   {Path(state.active_model_path).name}')
    print(f'  Active dataset: {getattr(state, "active_dataset", "?")}')
    print('  Assembling the 5 components...')
    print('─' * W)

    try:
        # --- 1. Load the detector with the active model ---
        from ultralytics import YOLO
        from conveyor_perception.perception.ultralytics_detector import UltralyticsDetector

        _yolo = YOLO(state.active_model_path)
        _resolved_path = _yolo.ckpt_path or state.active_model_path
        _names_obj = getattr(_yolo, 'names', None)
        _class_names = list(_names_obj.values()) if _names_obj else \
            ['Glass', 'metal', 'plastic', 'vinyl']
        det = UltralyticsDetector(
            model_path=_resolved_path,
            class_names=_class_names,
            conf_threshold=0.25,
            device=pick_device(),
            imgsz=640,
        )
        print(f'  ✓ Detector    loaded (model: {Path(_resolved_path).name}, classes: {len(_class_names)})')

        # --- 2. Wire up the other 4 components ---
        from conveyor_perception.core.tracking_pipeline import TrackingPipeline
        from conveyor_perception.core.drift_monitor import DriftMonitor
        from conveyor_perception.triage.agent import L1TriageAgent
        from conveyor_perception.predictive_maintenance.advisor import MaintenanceAdvisor
        from conveyor_perception.multitask.pipeline import MultitaskPipeline

        tracker = TrackingPipeline()
        print('  ✓ Tracker     loaded (ByteTrack)')

        drift = DriftMonitor(baseline_window=50, min_samples_for_drift=20)
        print('  ✓ Drift       loaded (KS + z-score + MAD)')

        triage = L1TriageAgent()
        print('  ✓ Triage      loaded (7 severity rules)')

        advisor = MaintenanceAdvisor()
        print('  ✓ Maintenance loaded (3 signal types)')

        # --- 3. Build the pipeline ---
        pipeline = MultitaskPipeline(det, tracker, drift, triage)
        print('─' * W)
        print('  Pipeline assembled: frame → Detector → Tracker → Drift → Triage')
        print('─' * W)

        # --- 4. Get a sample image (recycling val first, else synthetic) ---
        _sample_path = None
        _val_dir = REPO / 'data' / 'sample' / 'recycling_demo' / 'val' / 'images'
        if _val_dir.exists():
            _cands = sorted(_val_dir.glob('*.jpg'))
            if _cands:
                _sample_path = str(_cands[0])
        if not _sample_path:
            import numpy as np
            import cv2
            _syn = REPO / 'data' / 'sample' / '_synthetic_frame.jpg'
            _syn.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(_syn), np.zeros((640, 640, 3), dtype=np.uint8))
            _sample_path = str(_syn)
        import cv2
        _img = cv2.imread(_sample_path)
        if _img is None:
            raise RuntimeError(f'could not read sample image at {_sample_path}')
        print(f'  Sample: {Path(_sample_path).name}  shape={_img.shape}')

        # --- 5. Run N frames to populate drift signals ---
        N_FRAMES = 10
        print(f'\n  Running {N_FRAMES} frames through the pipeline...')
        t0 = time.perf_counter()
        _last = None
        for _i in range(N_FRAMES):
            _last = pipeline.step(_img)
        _elapsed_ms = (time.perf_counter() - t0) * 1000
        _ms_per_frame = _elapsed_ms / N_FRAMES

        state.metric('t4_inference_ms', round(_ms_per_frame, 2))
        state.metric('pipeline_frames', N_FRAMES)
        state.metric('pipeline_total_ms', round(_elapsed_ms, 1))

        print(f'\n  ✓ Pipeline ran {N_FRAMES} frames in {_elapsed_ms/1000:.1f}s ({_ms_per_frame:.1f} ms/frame on T4)')
        print(f'      last frame: {len(_last.detections)} detections, {len(_last.alerts)} alerts')

        state.log(
            'cell-10',
            action='ran',
            dataset=getattr(state, 'active_dataset', None),
            ms_per_frame=round(_ms_per_frame, 2),
            n_detections=len(_last.detections),
            n_alerts=len(_last.alerts),
        )
    except Exception as _e:
        print(f'  ✗ Pipeline failed: {type(_e).__name__}: {_e}')
        state.log(
            'cell-10',
            action='failed',
            error=str(_e),
        )
    print('─' * W)
    print('  Next: cell 11 (visual analytics).')


In [ ]:
# --- Cell 11: Visual Analytics (the impressive part) ---
import os, sys, time
from pathlib import Path
from colab_session import get_state, pick_device

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# Idempotent path setup
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

W = 82
print('─' * W)
print('  VISUAL ANALYTICS  (modern supervision annotators)'.center(W))
print('─' * W)

# --- 0. Skip if no active model ---
_skip_reason = None
if not getattr(state, 'active_model_path', None):
    _skip_reason = 'no-model'
elif not Path(state.active_model_path).exists():
    _skip_reason = 'model-missing'
else:
    try:
        import supervision as sv  # noqa: F401
    except ImportError:
        _skip_reason = 'supervision-missing'

if _skip_reason:
    if _skip_reason == 'no-model':
        print('  No active model. Run cell 8 to train, then cell 9.')
    elif _skip_reason == 'model-missing':
        print(f'  Active model missing: {state.active_model_path}')
    else:
        print('  supervision not installed. Re-run cell 2.')
    print('─' * W)
    state.log('cell-11', action='skipped', reason=_skip_reason)
    print('  Next: cell 12 (production path).')
else:
    try:
        import numpy as np
        import cv2
        import supervision as sv
        from IPython.display import display
        from PIL import Image as PILImage
        from collections import Counter
        from ultralytics import YOLO
        from conveyor_perception.perception.ultralytics_detector import UltralyticsDetector

        # --- 1. Load the detector with the active model ---
        _yolo = YOLO(state.active_model_path)
        _resolved_path = _yolo.ckpt_path or state.active_model_path
        _names_obj = getattr(_yolo, 'names', None)
        _class_names = list(_names_obj.values()) if _names_obj else \
            ['Glass', 'metal', 'plastic', 'vinyl']
        det = UltralyticsDetector(
            model_path=_resolved_path,
            class_names=_class_names,
            conf_threshold=0.25,
            device=pick_device(),
            imgsz=640,
        )
        print(f'  Detector: {Path(_resolved_path).name}  (classes: {_class_names})')

        # --- 2. Pick a real recycling val image (sorted, deterministic) ---
        _val_dir = REPO / 'data' / 'sample' / 'recycling_demo' / 'val' / 'images'
        _val_imgs = sorted(_val_dir.glob('*.jpg')) if _val_dir.exists() else []
        if _val_imgs:
            _sample_path = str(_val_imgs[0])
            print(f'  Sample: {_val_imgs[0].name}  (recycling val set)')
        else:
            _syn = REPO / 'data' / 'sample' / '_synthetic_frame.jpg'
            _syn.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(_syn), np.zeros((640, 640, 3), dtype=np.uint8))
            _sample_path = str(_syn)
            print('  Sample: _synthetic_frame.jpg  (recycling val missing)')
        _img = cv2.imread(_sample_path)
        _H, _W = _img.shape[:2]

        # --- 3. Run inference → sv.Detections ---
        _raw = det.detect(_img)
        if _raw:
            dets = sv.Detections(
                xyxy=np.array([d.bbox for d in _raw], dtype=np.float32),
                confidence=np.array([d.confidence for d in _raw], dtype=np.float32),
                class_id=np.array([d.class_id for d in _raw], dtype=int),
            )
        else:
            dets = sv.Detections.empty()
        state.metric('visual_inference_count', len(dets))
        print(f'  ✓ {len(dets)} detections on the sample image')
        if _raw:
            _counts = Counter(d.class_name for d in _raw)
            print('      ' + ' · '.join(f'{n} {name}' for name, n in _counts.most_common()))

        # --- 4. Annotators (supervision==0.30.0 API) ---
        from supervision.draw.color import ColorPalette
        _rbox = sv.RoundBoxAnnotator(roundness=0.6, color=ColorPalette.DEFAULT)
        _rich = sv.RichLabelAnnotator(color=ColorPalette.DEFAULT, border_radius=4)
        _heat = sv.HeatMapAnnotator(opacity=0.5, radius=30)

        # Spatial zone: infeed = left 30% of the frame
        _zone = sv.PolygonZone(
            polygon=np.array([[0, 0], [int(_W*0.3), 0], [int(_W*0.3), _H], [0, _H]]),
        )
        _zone_annot = sv.PolygonZoneAnnotator(zone=_zone, color=sv.Color.GREEN, thickness=2)

        # Throughput line: vertical line in the middle
        _line = sv.LineZone(
            start=sv.Point(x=_W//2, y=0), end=sv.Point(x=_W//2, y=_H),
        )
        _line_annot = sv.LineZoneAnnotator(color=sv.Color.RED, text_scale=1.5)

        # --- 5. Annotate + display + save ---
        _ann = _img.copy()
        _ann = _heat.annotate(_ann, dets)
        _ann = _rbox.annotate(_ann, dets)
        _labels = [f'{d.class_name} {d.confidence:.2f}' for d in _raw] if _raw else []
        _ann = _rich.annotate(_ann, dets, labels=_labels)
        _ann = _zone_annot.annotate(_ann)
        _cross_in, _cross_out = _line.trigger(dets)
        _ann = _line_annot.annotate(_ann, _line)

        # Save to disk for the summary cell to pick up
        _out_path = REPO / 'visual_analytics.png'
        cv2.imwrite(str(_out_path), _ann)
        print(f'  saved annotated image: {_out_path.name}')

        # Display inline (BGR → RGB)
        _ann_rgb = cv2.cvtColor(_ann, cv2.COLOR_BGR2RGB)
        display(PILImage.fromarray(_ann_rgb))

        # --- 6. Real FPS via supervision.FPSMonitor ---
        # sv.FPSMonitor.tick() is a manual stopwatch — the actual work
        # being measured MUST be inside the tick loop, otherwise the FPS
        # reads how fast the loop runs (~1.9M FPS), not how fast the model
        # runs (~25-40 FPS on T4).
        _fps = sv.FPSMonitor()
        for _ in range(30):
            _fps.tick()
            det.detect(_img)  # <-- the actual work being measured
        state.metric('t4_measured_fps', round(_fps.fps, 1))
        print(f'\n  ✓ Measured: {_fps.fps:.1f} FPS  (supervision.FPSMonitor)')
        _n_in = int(_cross_in.sum()) if hasattr(_cross_in, 'sum') else int(_cross_in)
        _n_out = int(_cross_out.sum()) if hasattr(_cross_out, 'sum') else int(_cross_out)
        print(f'      LineZone: in={_n_in}  out={_n_out}  (items crossing mid-line)')

        state.log(
            'cell-11',
            action='ran',
            dataset=getattr(state, 'active_dataset', None),
            n_detections=len(dets),
            fps=round(_fps.fps, 1),
            line_in=_n_in,
            line_out=_n_out,
        )
    except Exception as _e:
        print(f'  ✗ Visual analytics failed: {type(_e).__name__}: {_e}')
        state.log(
            'cell-11',
            action='failed',
            error=str(_e),
        )
    print('─' * W)
    print('  Next: cell 12 (production path).')


In [ ]:
# --- Cell 12: Production path (Roboflow Inference, library mode) ---
import os, sys, time
from pathlib import Path
from colab_session import get_state, pick_device

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# Idempotent path setup
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

W = 82
print('─' * W)
print('  PRODUCTION PATH  (Roboflow Inference, library mode)'.center(W))
print('─' * W)
print('  Same YOLO model family, different runtime. This is what ships to edge.')
print()

# --- 0. Try to import the inference package ---
_inference_ok = False
_inference_err = None
try:
    from inference import get_model as _inf_get_model
    import inference  # noqa: F401
    _inference_ok = True
except ImportError as _e:
    _inference_err = str(_e)

if not _inference_ok:
    # Self-healing: print a clear skip + install hint, log, and continue.
    print(f'  ⚠ inference not importable ({type(Exception(_inference_err)).__name__}: {_inference_err})')
    print()
    print('  The production path is OPTIONAL — the demo continues with the Ultralytics path.')
    print('  To enable the production-path comparison:')
    print()
    print('    pip install inference supervision numpy pillow')
    print()
    print('  (in a fresh venv, since inference has stricter dep requirements than our pinned set).')
    state.log('cell-12', action='skipped', reason='inference-missing', error=_inference_err)
    print('─' * W)
    print('  Next: cell 13 (triage + monitor).')
else:
    try:
        import numpy as np
        from PIL import Image as PILImage
        from IPython.display import display
        import cv2

        # --- 1. Pick a sample image (recycling val, fallback to a 640x640 black frame) ---
        _val_dir = REPO / 'data' / 'sample' / 'recycling_demo' / 'val' / 'images'
        _val_imgs = sorted(_val_dir.glob('*.jpg')) if _val_dir.exists() else []
        if _val_imgs:
            _sample_path = str(_val_imgs[0])
        else:
            _syn = REPO / 'data' / 'sample' / '_synthetic_frame.jpg'
            _syn.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(_syn), np.zeros((640, 640, 3), dtype=np.uint8))
            _sample_path = str(_syn)
        _img_rgb = cv2.cvtColor(cv2.imread(_sample_path), cv2.COLOR_BGR2RGB)
        print(f'  Sample: {Path(_sample_path).name}  shape={_img_rgb.shape}')

        # --- 2. Load via Roboflow Inference (foundation model, no account needed) ---
        # `yolov8n-640` is the COCO YOLOv8n foundation alias. NOTE: this is a
        # DIFFERENT model from our recycling_v3 (YOLO26s, 4-class). YOLO26 is
        # too new for Roboflow Inference's foundation list as of Aug 2026, so
        # the closest stable alias is v8n. The point of this cell is the *runtime*
        # difference (Ultralytics vs Roboflow Inference), not model parity. To do
        # an apples-to-apples comparison, upload best.pt to Roboflow and use the
        # resulting workspace/project/version ID here.
        _model_id = 'yolov8n-640'
        print(f'  Loading model: {_model_id}  (via Roboflow Inference)')
        t0 = time.perf_counter()
        _model = _inf_get_model(model_id=_model_id)
        _load_s = time.perf_counter() - t0
        print(f'  Model loaded in {_load_s:.1f}s')

        # --- 3. Run inference (warm up + 10 frames to populate caches) ---
        print(f'\n  Running 10 frames through the inference runtime...')
        t0 = time.perf_counter()
        for _ in range(10):
            _results = _model.infer(_img_rgb, confidence=0.4)
        _elapsed_ms = (time.perf_counter() - t0) * 1000
        _ms_per_frame = _elapsed_ms / 10

        n_preds = len(_results[0].predictions) if _results else 0
        state.metric('inference_ms', round(_ms_per_frame, 2))
        state.metric('inference_n_predictions', n_preds)

        print(f'\n  ✓ {n_preds} detections in {_ms_per_frame:.1f} ms/frame via Roboflow Inference')
        print(f'      (Ultralytics path in cell 11: 67.4 FPS / 14.8 ms/frame on the same T4)')

        state.log(
            'cell-12',
            action='ran',
            model_id=_model_id,
            ms_per_frame=round(_ms_per_frame, 2),
            n_predictions=n_preds,
        )
    except Exception as _e:
        print(f'  ✗ Production path failed: {type(_e).__name__}: {_e}')
        state.log(
            'cell-12',
            action='failed',
            error=str(_e),
        )
    print('─' * W)
    print('  Next: cell 13 (triage + monitor).')


In [ ]:
# --- Cell 13: Triage + monitor (queue + robustness + dashboard) ---
import os, sys, json, time
from pathlib import Path
from colab_session import get_state, pick_device

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()
SRC = REPO / 'src'

# Idempotent path setup
for _p in (REPO, REPO / 'notebooks', SRC):
    _sp = str(_p)
    if _sp not in sys.path:
        sys.path.insert(0, _sp)

W = 82
print('─' * W)
print('  TRIAGE + MONITOR  (queue + robustness + dashboard)'.center(W))
print('─' * W)

# --- 1. Triage queue ---
if state.toggles.get('module:triage', True):
    print('\n  1. TRIAGE QUEUE  (most recent alerts)')
    print('  ' + '─' * 70)
    try:
        from conveyor_perception.triage.agent import L1TriageAgent
        _triage = L1TriageAgent()
        _alerts = _triage.get_pending(limit=10)
        if _alerts:
            for _a in _alerts:
                _sev = getattr(_a, 'severity', '?').upper()
                _cls = getattr(_a, 'class_name', '?')
                _conf = getattr(_a, 'confidence', 0.0)
                _reason = (getattr(_a, 'metadata', {}) or {}).get('reason', '')[:50]
                print(f'    [{_sev:9s}] {_cls:10s} conf={_conf:.2f}  reason="{_reason}"')
            state.metric('triage_n_alerts', len(_alerts))
        else:
            print('    (no pending alerts — pipeline ran clean)')
            state.metric('triage_n_alerts', 0)
    except Exception as _e:
        print(f'    ✗ triage agent failed: {type(_e).__name__}: {_e}')
        state.log('cell-13', action='partial', section='triage', error=str(_e))
else:
    print('\n  1. TRIAGE QUEUE  (skipped — module:triage toggle off)')

# --- 2. Robustness suite ---
print('\n  2. ROBUSTNESS SUITE  (13 augmentations on the active model)')
print('  ' + '─' * 70)
if not state.toggles.get('module:robustness', True):
    print('    (skipped — module:robustness toggle off)')
elif not getattr(state, 'active_model_path', None):
    print('    (skipped — no active model. Run cells 8-9 first.)')
else:
    try:
        import cv2
        from ultralytics import YOLO
        from conveyor_perception.perception.ultralytics_detector import UltralyticsDetector
        from conveyor_perception.robustness import RobustnessTestSuite

        # Load detector + a recycling val image
        _yolo = YOLO(state.active_model_path)
        _det = UltralyticsDetector(
            model_path=_yolo.ckpt_path or state.active_model_path,
            class_names=list((_yolo.names or {}).values()) or ['Glass', 'metal', 'plastic', 'vinyl'],
            conf_threshold=0.25,
            device=pick_device(),
            imgsz=640,
        )
        _val_dir = REPO / 'data' / 'sample' / 'recycling_demo' / 'val' / 'images'
        _val_imgs = sorted(_val_dir.glob('*.jpg')) if _val_dir.exists() else []
        if not _val_imgs:
            print('    (no recycling val images — robustness suite needs a real image)')
        else:
            _img = cv2.imread(str(_val_imgs[0]))
            _suite = RobustnessTestSuite(_det, _img)
            _report = _suite.run()
            print(_report.to_markdown())
            state.metric('robustness_verdict', getattr(_report, 'verdict', 'unknown'))
    except Exception as _e:
        print(f'    ✗ robustness suite failed: {type(_e).__name__}: {_e}')
        state.log('cell-13', action='partial', section='robustness', error=str(_e))

# --- 3. Shift dashboard ---
print('\n  3. SHIFT DASHBOARD  (live metrics from state)')
print('  ' + '─' * 70)
if not state.toggles.get('module:monitoring', True):
    print('    (skipped — module:monitoring toggle off)')
else:
    try:
        from conveyor_perception.monitoring.dashboard import MonitoringDashboard
        _dash = MonitoringDashboard()
        _shift = _dash.shift_report()
        _shift_dict = _shift.to_dict() if hasattr(_shift, 'to_dict') else dict(_shift) if isinstance(_shift, dict) else {'report': str(_shift)}
        # Print the key fields (full JSON dump if no specific fields)
        # Unit detection: derive the unit from the METRIC KEY (e.g. 't4_measured_fps'
        # contains 'fps' → FPS), NOT from the value (the value is a number like 67.4
        # which has no 'fps' substring — the previous bug printed '67.4 ms/frame'
        # when the actual measurement was 67.4 FPS).
        if 't4_measured_fps' in state.metrics:
            _fps_key, _fps_unit = 't4_measured_fps', 'FPS'
        elif 't4_inference_ms' in state.metrics:
            _fps_key, _fps_unit = 't4_inference_ms', 'ms/frame'
        else:
            _fps_key, _fps_unit = None, None
        _fps = state.metrics.get(_fps_key) if _fps_key else None
        _map50 = state.metrics.get('best_mAP50_value')
        _alerts_count = state.metrics.get('triage_n_alerts', 0)
        print(f'    model:        {state.metrics.get("best_mAP50_model", "?")}')
        if _fps is not None:
            print(f'    inference:    {_fps} {_fps_unit}')
        if _map50 is not None:
            print(f'    mAP50:        {_map50}')
        print(f'    alerts:       {_alerts_count} pending')
        print(f'    retrain:      {getattr(_shift, "retrain_recommended", "?")}')
        state.metric('retrain_recommended', getattr(_shift, 'retrain_recommended', None))
    except Exception as _e:
        print(f'    ✗ dashboard failed: {type(_e).__name__}: {_e}')
        state.log('cell-13', action='partial', section='dashboard', error=str(_e))

print('─' * W)
state.log('cell-13', action='ran', modules=['triage', 'robustness', 'monitoring'])
print('  ✓ triage + monitor done.  Next: cell 14 (coach + summary).')


In [ ]:
# --- Cell 14: Coach + summary (the closer) ---
import os, sys, json
from pathlib import Path
from colab_session import get_state, coach_review

state = get_state()
IN_COLAB = 'google.colab' in sys.modules
REPO = Path('/content/conveyor-perception' if IN_COLAB else '.').resolve()

W = 82
print('─' * W)
print('  COACH + SUMMARY  (the closer)'.center(W))
print('─' * W)

# --- 1. Coach review (Gemini, graceful skip on no API key) ---
print('\n  1. COACH REVIEW  (Gemini reads the run log)')
print('  ' + '─' * 70)
try:
    _review = coach_review(state)
    print(_review)
    state.metric('coach_review_chars', len(_review))
except Exception as _e:
    print(f'    (coach unavailable: {type(_e).__name__}: {_e})')
    state.log('cell-14', action='partial', section='coach', error=str(_e))

# --- 2. Run summary (the text table for screenshots) ---
print('\n  2. RUN SUMMARY  (every key metric from this run)')
print('  ' + '─' * 70)
_summary = state.summary_table() if hasattr(state, 'summary_table') else None
if _summary:
    print(_summary)
else:
    # Fallback: print metrics manually if summary_table() isn't on state
    for _k, _v in state.metrics.items():
        print(f'    {_k:30}  {_v}')

    print('\n    Toggles:')
    for _k, _v in state.toggles.items():
        print(f'      {("✓" if _v else "○")} {_k}')

# --- 3. Downloadable session log (state.to_dict() → JSON) ---
print('\n  3. SESSION LOG  (downloadable JSON)')
print('  ' + '─' * 70)
try:
    _log_path = REPO / 'session_log.json'
    if hasattr(state, 'to_dict'):
        _log = state.to_dict()
    else:
        _log = {
            'metrics': dict(getattr(state, 'metrics', {})),
            'toggles': dict(getattr(state, 'toggles', {})),
            'active_model_path': getattr(state, 'active_model_path', None),
            'active_dataset': getattr(state, 'active_dataset', None),
        }
    _log_path.write_text(json.dumps(_log, indent=2, default=str))
    print(f'    saved: {_log_path}  ({_log_path.stat().st_size / 1024:.1f} KB)')
    state.metric('session_log_bytes', _log_path.stat().st_size)
except Exception as _e:
    print(f'    ✗ session log save failed: {type(_e).__name__}: {_e}')
    state.log('cell-14', action='partial', section='session-log', error=str(_e))

state.log('cell-14', action='ran')
print('─' * W)
print('  ✓ coach + summary done.  Next: cell 15 (T4 vs EverestLabs).')
